# 🔭 Notebook 1: The Three Pillars — Logs, Metrics, Traces

Imagine you run a website and someone tweets *"the checkout page is broken!"*. You jump to your terminal — what do you look at?

Real production systems use three complementary tools:

- 🪵 **Logs** — *"what happened, in detail, in chronological order?"* — best for debugging a single request.
- 📊 **Metrics** — *"how is the system doing right now, in aggregate?"* — best for dashboards and alerts.
- 🧵 **Traces** — *"where did this one user request spend its time across services?"* — best for finding latency bottlenecks.

Each is bad at the others' job. Together they form **observability**.

## Learning objectives
- Generate fake logs, metrics, and traces with nothing but `print` and dicts.
- See why you need all three to debug a real outage.

## 🛠️ Setup

```bash
cd 01-foundations/observability
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). Reload the window if it doesn't appear.

## 🪵 Logs — the diary

A log line is timestamped text. Good for "what exactly happened to *this* request?" — bad for "how is the system doing across millions of requests?" because reading them all is slow.

Tip: use **structured logs** (key=value or JSON) so they are searchable later.

In [ ]:
import time, json, uuid, random

def log(level, event, **fields):
    record = {
        "ts": time.strftime("%H:%M:%S"),
        "level": level,
        "event": event,
        **fields,
    }
    print(json.dumps(record))

req_id = uuid.uuid4().hex[:6]
log("INFO",  "request.start", req_id=req_id, path="/checkout")
log("INFO",  "db.query",      req_id=req_id, table="orders", ms=12)
log("ERROR", "payment.failed", req_id=req_id, reason="card_declined")
log("INFO",  "request.end",   req_id=req_id, status=402, ms=87)

## 📊 Metrics — the dashboard

A metric is a single number sampled over time. There are usually three kinds:

- **Counter** — only goes up (e.g. `http_requests_total`).
- **Gauge** — goes up and down (e.g. `memory_in_use_bytes`).
- **Histogram** — buckets for latency (e.g. `http_request_duration_ms`).

Metrics are tiny, cheap, and fast to query — perfect for alerts. They cannot tell you *which* request was slow, just that 5% of them were.

In [ ]:
class Metrics:
    def __init__(self):
        self.counters = {}
        self.gauges = {}
        self.histograms = {}

    def inc(self, name, by=1):
        self.counters[name] = self.counters.get(name, 0) + by

    def gauge(self, name, value):
        self.gauges[name] = value

    def observe(self, name, value):
        self.histograms.setdefault(name, []).append(value)

m = Metrics()
random.seed(0)
for _ in range(1000):
    latency = random.gauss(50, 10)
    m.observe("http_latency_ms", latency)
    m.inc("http_requests_total")
    if latency > 70:
        m.inc("http_requests_slow_total")

m.gauge("memory_in_use_mb", 412)

vals = sorted(m.histograms["http_latency_ms"])
p50, p95, p99 = vals[500], vals[950], vals[990]
print(f"requests: {m.counters['http_requests_total']}")
print(f"slow:     {m.counters['http_requests_slow_total']}")
print(f"p50={p50:.1f} ms  p95={p95:.1f} ms  p99={p99:.1f} ms")
print(f"memory:   {m.gauges['memory_in_use_mb']} MB")

## 🧵 Traces — the breadcrumb trail

A **trace** follows one request as it bounces between services. Each step is a **span**: a name, a start time, a duration, and a parent span. Together they form a tree.

Traces are great for "where is the time going?" in a microservices architecture.

In [ ]:
class Span:
    def __init__(self, name, parent=None):
        self.name = name
        self.parent = parent
        self.start = time.perf_counter()
        self.children = []
        if parent:
            parent.children.append(self)

    def __enter__(self): return self
    def __exit__(self, *a): self.duration_ms = (time.perf_counter() - self.start) * 1000

    def show(self, indent=0):
        print(f"{' ' * indent}└─ {self.name}  ({self.duration_ms:.1f} ms)")
        for c in self.children:
            c.show(indent + 4)

with Span("GET /checkout") as root:
    time.sleep(0.005)
    with Span("auth.verify_jwt", parent=root):
        time.sleep(0.002)
    with Span("orders.create", parent=root) as orders:
        with Span("db.insert", parent=orders):
            time.sleep(0.020)
        with Span("payments.charge", parent=orders):
            time.sleep(0.080)         # the slow one!
    with Span("send_email", parent=root):
        time.sleep(0.003)

root.show()

## 🤔 When to use which

| Question | Best tool |
|---|---|
| Is the system healthy right now? Are we in SLO? | **Metrics** |
| Why was *this specific request* slow / wrong? | **Traces** |
| What was the exact error / stack trace for request X? | **Logs** |

The big trick: include the same `request_id` in your logs **and** as a trace id, so you can jump between them when debugging.